In [ ]:
def _add_arcsinh_price_lags(df: pd.DataFrame) -> pd.DataFrame:
    """
    arcsinh-transformed price lags.
    arcsinh handles negatives and compresses extreme spikes,
    giving the model a better-scaled view of price history.
    """
    df   = df.copy()
    scale = 100
    ap   = np.arcsinh(df["price"] / scale).rename("_ap")
    for lag in [1, 2, 4, 12, 48, 96, 336, 335, 337]:
        df[f"price_asinh_lag_{lag}"] = ap.shift(lag).astype(np.float32)
    df["price_asinh_rmean_48"]  = ap.rolling(48).mean().astype(np.float32)
    df["price_asinh_rmean_336"] = ap.rolling(336).mean().astype(np.float32)
    return df

In [ ]:
def _add_time_features(df: pd.DataFrame) -> pd.DataFrame:

    _NSW_HOLIDAYS = holidays.Australia(state="NSW", years=range(2018, 2025))
    
    idx = df.index
    df = df.copy()

    # Raw calendar components
    df["hour"]        = idx.hour.astype(np.int8)
    df["dayofweek"]   = idx.dayofweek.astype(np.int8)
    df["month"]       = idx.month.astype(np.int8)
    df["dayofyear"]   = idx.day_of_year.astype(np.int16)

    # Cyclical (sin/cos) encodings so the model sees periodicity
    df["hour_sin"]    = np.sin(2 * np.pi * idx.hour / 24).astype(np.float32)
    df["hour_cos"]    = np.cos(2 * np.pi * idx.hour / 24).astype(np.float32)
    df["dow_sin"]     = np.sin(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    df["dow_cos"]     = np.cos(2 * np.pi * idx.dayofweek / 7).astype(np.float32)
    df["month_sin"]   = np.sin(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)
    df["month_cos"]   = np.cos(2 * np.pi * (idx.month - 1) / 12).astype(np.float32)

    # Binary flags
    df["is_weekend"]  = (idx.dayofweek >= 5).astype(np.float32)
    df["is_holiday"]  = np.array(
        [d.date() in _NSW_HOLIDAYS for d in idx], dtype=np.float32
    )
    # Peak (17–20 h) and off-peak (< 7 h or ≥ 21 h) periods
    df["is_peak"]     = ((idx.hour >= 17) & (idx.hour <= 20)).astype(np.float32)
    df["is_shoulder"] = ((idx.hour >= 7)  & (idx.hour < 17)).astype(np.float32)
    df["is_off_peak"] = ((idx.hour < 7)   | (idx.hour >= 21)).astype(np.float32)

    # Combined weekend/holiday flag — demand behaviour is quite different
    df["is_offday"]   = np.maximum(df["is_weekend"], df["is_holiday"]).astype(np.float32)

    return df


In [ ]:
def _add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    LAG_INTERVALS = [
        1, 2, 3, 4, 6, 8, 12, 24,
        48, 49, 50, 51, 52,
        96, 97, 98,
        # 3-day and 4-day same-period anchors — captures weekly envelope effects
        # without redundancy with the weekly (336) lag.
        144, 143, 145,
        192, 191, 193,
        336, 335, 337,
        672, 671, 673,
    ]
    
    df = df.copy()
    for lag in sorted(set(LAG_INTERVALS)):
        df[f"price_lag_{lag}"]  = df["price"].shift(lag).astype(np.float32)
    
    for lag in range(26, 72, 2):         # 26, 28, ..., 70 (step = 1 h = 2 intervals)
        if lag not in LAG_INTERVALS:
            df[f"price_lag_{lag}"] = df["price"].shift(lag).astype(np.float32)
    return df


In [ ]:
def _add_long_range_features(df: pd.DataFrame) -> pd.DataFrame:
    
    df   = df.copy()
    p    = df["price"]
    ap   = np.arcsinh(p / 100)
    BASE = 17_532

    # --- Annual price lags (same period last year ± spread) ---
    for offset in range(-2, 2 + 1):
        lag = BASE + offset
        df[f"price_lag_annual_{'+' if offset >= 0 else ''}{offset}"] = (
            p.shift(lag).astype(np.float32)
        )
        df[f"price_asinh_lag_annual_{'+' if offset >= 0 else ''}{offset}"] = (
            ap.shift(lag).astype(np.float32)
        )

    p_annual = p.shift(BASE)
    df["price_annual_rmean_96"]  = p_annual.rolling(96, min_periods=24).mean().astype(np.float32)
    df["price_annual_rmax_96"]   = p_annual.rolling(96, min_periods=24).max().astype(np.float32)
    df["price_annual_rstd_96"]   = p_annual.rolling(96, min_periods=24).std().astype(np.float32)

    # Spike count over the same week last year
    df["price_annual_spike_96"]  = (p_annual >= 300).rolling(96, min_periods=24).sum().astype(np.float32)

    # Year-on-year price change (current vs same period last year)
    df["price_yoy_change"]  = (p - p.shift(BASE)).astype(np.float32)
    df["price_yoy_ratio"]   = (p / (p.shift(BASE).abs() + 1)).clip(0, 20).astype(np.float32)

    # --- 2-week rolling statistics (not in main ROLLING_WINDOWS to keep loop tight) ---
    df["price_rmean_2w"]  = p.rolling(672, min_periods=336).mean().astype(np.float32)
    df["price_rmax_2w"]   = p.rolling(672, min_periods=336).max().astype(np.float32)
    df["price_rmin_2w"]   = p.rolling(672, min_periods=336).min().astype(np.float32)
    df["price_rstd_2w"]   = p.rolling(672, min_periods=336).std().astype(np.float32)

    # 6-week rolling mean (seasonal trend)
    df["price_rmean_6w"]  = p.rolling(2016, min_periods=1008).mean().astype(np.float32)

    return df


In [ ]:
def _add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Rolling statistics computed on price available at each timestamp."""
    ROLLING_WINDOWS = [4, 8, 24, 48, 96, 336, 672, 2016]

    df = df.copy()
    p = df["price"]
    for w in ROLLING_WINDOWS:
        min_p = max(1, w // 2)
        rolled = p.rolling(w, min_periods=min_p)
        df[f"price_rmean_{w}"] = rolled.mean().astype(np.float32)
        df[f"price_rstd_{w}"]  = rolled.std().astype(np.float32)
        df[f"price_rmax_{w}"]  = rolled.max().astype(np.float32)
        df[f"price_rmin_{w}"]  = rolled.min().astype(np.float32)
    return df


In [ ]:
def _add_regime_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Capture price-spike and volatility regime signals.
    All windows look backward only, so there is no future leakage.
    """
    df = df.copy()
    p  = df["price"]

    # Was there a high-price spike in the last 24 h?
    df["spike_flag_48"]    = (p.rolling(48).max()  > 300).astype(np.float32)
    # Was there a negative price in the last 24 h?
    df["neg_flag_48"]      = (p.rolling(48).min()  < 0).astype(np.float32)
    # Recent volatility
    df["price_vol_48"]     = p.rolling(48).std().astype(np.float32)
    df["price_vol_336"]    = p.rolling(336).std().astype(np.float32)
    # Quantile context (recent price level relative to weekly range)
    df["price_q90_336"]    = p.rolling(336).quantile(0.90).astype(np.float32)
    df["price_q10_336"]    = p.rolling(336).quantile(0.10).astype(np.float32)
    df["price_pct_rank_48"] = (
        p.rolling(48).rank(pct=True)
    ).astype(np.float32)

    return df


In [ ]:
def _add_time_since_spike_features(df: pd.DataFrame) -> pd.DataFrame:
   
    df   = df.copy()
    p    = df["price"]
    # 30-min intervals between rows
    INTERVAL_H = 0.5
    # Maximum hours to report (cap at 2 weeks to avoid unbounded values early in series)
    MAX_HOURS  = 336.0  # 2 weeks

    for threshold in [150, 300, 1000]:
        spike_flag = (p >= threshold).astype(np.float32)

        cumsum   = spike_flag.cumsum()
       
        positions       = pd.RangeIndex(len(df))
        last_spike_pos  = (
            pd.Series(np.where(spike_flag.values, positions, np.nan), index=df.index)
            .ffill()
            .fillna(-MAX_HOURS / INTERVAL_H)  # no prior spike seen → treat as max
        )
        intervals_since = (pd.Series(positions, index=df.index) - last_spike_pos).clip(upper=MAX_HOURS / INTERVAL_H)
        hours_since     = (intervals_since * INTERVAL_H).astype(np.float32)
        col             = f"hours_since_spike_{threshold}"
        df[col]         = hours_since

        # Log-scale version reduces the numeric range for the tree learner
        df[f"log1p_hours_since_spike_{threshold}"] = np.log1p(hours_since).astype(np.float32)

        # Binary flags: was there a spike at each lookback horizon?
        for lookback_h in [1, 6, 12, 24, 48, 168]:   # 1h, 6h, 12h, 24h, 48h, 1wk
            intervals = int(lookback_h / INTERVAL_H)
            df[f"spike_{threshold}_last_{lookback_h}h"] = (
                spike_flag.rolling(intervals, min_periods=1).max().astype(np.float32)
            )

    # Combined: hours since any of the three thresholds (summarises overall stress state)
    df["hours_since_any_spike"] = df[["hours_since_spike_150",
                                      "hours_since_spike_300",
                                      "hours_since_spike_1000"]].min(axis=1).astype(np.float32)

    return df


In [ ]:
def _add_spike_predictors(df: pd.DataFrame) -> pd.DataFrame:
    """
    Price momentum and spike-history features.
    All look-back only — zero leakage.
    """
    df = df.copy()
    p  = df["price"]

    # Price momentum — how fast prices are moving right now
    df["price_mom_4"]  = p.diff(4).astype(np.float32)    # 2h
    df["price_mom_12"] = p.diff(12).astype(np.float32)   # 6h
    df["price_mom_48"] = p.diff(48).astype(np.float32)   # 24h

    # Price acceleration (second derivative) — is the current spike accelerating?
    df["price_accel_4"]  = p.diff(4).diff(4).clip(-2000, 2000).astype(np.float32)
    df["price_accel_12"] = p.diff(12).diff(12).clip(-2000, 2000).astype(np.float32)

    # Spike and negative price counts in recent history
    df["spike_count_48"]  = (p >= 300).rolling(48).sum().astype(np.float32)
    df["spike_count_336"] = (p >= 300).rolling(336).sum().astype(np.float32)
    df["neg_count_48"]    = (p < 0).rolling(48).sum().astype(np.float32)

    # Spike intensity: cumulative spike energy in last 24h (not just count)
    SPIKE_THR = 150.0
    spike_excess = (p - SPIKE_THR).clip(lower=0)
    df["spike_intensity_48"]  = spike_excess.rolling(48).sum().astype(np.float32)
    df["spike_intensity_336"] = spike_excess.rolling(336).sum().astype(np.float32)

    # Price percentile rank within recent window (how extreme is current price?)
    df["price_pctrank_48"]  = p.rolling(48).rank(pct=True).astype(np.float32)
    df["price_pctrank_336"] = p.rolling(336).rank(pct=True).astype(np.float32)

    return df


In [ ]:
def _add_region_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lag and rolling features for neighbouring NEM region prices (QLD, VIC, SA).
    Price spreads between regions indicate interconnector pressure — a
    key leading indicator of NSW spikes. SA1 is the most volatile NEM
    region and is a leading indicator of gas-driven spikes that propagate
    through the VIC-NSW interconnector.
    """
    df = df.copy()
    new_cols: dict = {}
    for col in ["qld_price", "vic_price", "sa_price"]:
        if col not in df.columns:
            continue
        p = df[col]
        # Level lags
        for lag in [1, 2, 4, 48, 96, 336]:
            new_cols[f"{col}_lag_{lag}"] = p.shift(lag).astype(np.float32)
        # Rolling means
        for w in [4, 24, 48]:
            new_cols[f"{col}_rmean_{w}"] = p.rolling(w).mean().astype(np.float32)
        # Recent spike count in neighbour region
        new_cols[f"{col}_spike_48"] = (p >= 300).rolling(48).sum().astype(np.float32)
        # arcsinh-transformed lags (handles negatives + compresses spikes)
        ap = np.arcsinh(p / 100)
        for lag in [1, 48, 336]:
            new_cols[f"{col}_asinh_lag_{lag}"] = ap.shift(lag).astype(np.float32)
        # Spread vs NSW price (interconnector pressure direction)
        spread = (p - df["price"]).astype(np.float32)
        new_cols[f"{col}_spread"]      = spread
        new_cols[f"{col}_spread_lag1"] = spread.shift(1).astype(np.float32)
        # Rolling max — SA extreme spikes are a strong warning signal
        new_cols[f"{col}_rmax_48"]  = p.rolling(48).max().astype(np.float32)
        new_cols[f"{col}_rmax_336"] = p.rolling(336).max().astype(np.float32)

    # Multi-region spike co-occurrence: all three neighbours elevated simultaneously
    regions_present = [c for c in ["qld_price", "vic_price", "sa_price"] if c in df.columns]
    if len(regions_present) >= 2:
        flags = pd.concat([(df[c] >= 150).astype(np.float32) for c in regions_present], axis=1)
        new_cols["multi_region_spike"] = flags.min(axis=1)  # 1 only if ALL elevated
        new_cols["region_spike_count"] = flags.sum(axis=1).astype(np.float32)  # 0-3

    if new_cols:
        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return df
